# 3b — What is vector search? Semantic search from scratch

Companion notebook to blog post **3b (What is vector search?)**. Runs standalone —
the BM25 cell below is post 2a's scorer, included so you can watch it fail first.

In [ ]:
%pip install -q fastembed numpy qdrant-client

## Where BM25 hits its wall

Post 2a's full scorer, same five sentences, query `puppy playing outside` — every score 0.000,
even though two documents are literally about dogs playing outside.

In [ ]:
import math
from collections import Counter

def tokenize(text):
    return text.lower().split()

corpus = [
    "The cat sat on the warm windowsill in the sun",       # doc 0
    "A dog chased the cat around the yard",                 # doc 1
    "Dogs are loyal and love to play fetch in the park",    # doc 2
    "The park has a pond where ducks swim every morning",   # doc 3
    "She planted tomatoes and basil in her garden",         # doc 4
]

docs = [tokenize(d) for d in corpus]
N = len(docs)                              # 5 documents
avgdl = sum(len(d) for d in docs) / N      # average document length

# document frequency: in how many documents does each word appear?
df = Counter()
for d in docs:
    for term in set(d):
        df[term] += 1

def idf(term):
    n = df.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def term_score(term, doc, k1=1.5, b=0.75):
    freqs = Counter(doc)
    if term not in freqs:
        return 0.0
    f = freqs[term]
    dl = len(doc)
    doc_length_norm = 1 - b + b * dl / avgdl
    term_freq_saturation = f * (k1 + 1) / (f + k1 * doc_length_norm)
    return idf(term) * term_freq_saturation

def bm25_score(query, doc):
    return sum(term_score(t, doc) for t in tokenize(query))

print("Query: 'puppy playing outside'")
for i, d in enumerate(docs):
    print(f"  doc {i}: {bm25_score('puppy playing outside', d):.3f}")

## Idea 1 — Embeddings: give every text coordinates

Toy version first: describe any text with two hand-made numbers (animal-ness, plant-ness).
Position captures meaning — `dog` and `puppy` land together, `tomato` far away.

In [ ]:
vec = {
    "dog":    [0.9, 0.1],   # very animal, barely plant
    "puppy":  [0.8, 0.1],   # very animal, barely plant
    "tomato": [0.1, 0.9],   # barely animal, very plant
}

## Idea 2 — Cosine similarity: measure closeness

Head math with `a = [3, 4]`, `b = [6, 8]` (same direction, twice as long):

- `dot(a, b) = 3×6 + 4×8 = 50` — big, but big because similar or because long?
- `norm(a) = √(3²+4²) = 5`, `norm(b) = 10` (Pythagoras — the arrow's length)
- `cosine = 50 / (5 × 10) = 1.0` — divide and the length cancels; only DIRECTION survives

Double `b` to `[12,16]`: dot doubles to 100, norm doubles to 20 → cosine still 1.0.

In [ ]:
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

a, b, c = [3, 4], [6, 8], [4, -3]
print("dot(a,b) =", np.dot(a, b), "  norms:", np.linalg.norm(a), np.linalg.norm(b))
print("cosine(a, b)       =", cosine(a, b))          # same direction -> 1.0
print("cosine(a, [12,16]) =", cosine(a, [12, 16]))   # longer, same direction -> still 1.0
print("cosine(a, c)       =", cosine(a, c))          # perpendicular -> 0.0
print()
print("toy vectors:")
print("cosine(dog, puppy)  =", round(cosine(vec["dog"], vec["puppy"]), 3))
print("cosine(dog, tomato) =", round(cosine(vec["dog"], vec["tomato"]), 3))

## A real embedding model

`all-MiniLM-L6-v2` — 384 numbers per text, runs locally, free. No single dimension is
readable; meaning lives in the position as a whole.

In [ ]:
from fastembed import TextEmbedding

model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

emb = list(model.embed(["A dog chased the cat around the yard"]))[0]
print("length:", len(emb))
print("first 6 numbers:", [round(float(v), 3) for v in emb[:6]])

The moment keyword search could never reach — synonyms, no synonym list anywhere:

In [ ]:
words = ["puppy", "dog", "tomato", "car", "automobile"]
wembs = dict(zip(words, model.embed(words)))
# wembs maps each word to its 384-number vector

for x, y in [("puppy", "dog"), ("puppy", "tomato"), ("car", "automobile"), ("car", "tomato")]:
    print(f"cosine({x!r:12}, {y!r:12}) = {cosine(wembs[x], wembs[y]):.3f}")

## Idea 3 — Search = nearest neighbors on the map

Embed every document once, embed the query, rank by cosine. Every query below scores
0.000 across the board under BM25 — watch the map instead.

In [ ]:
doc_embs = list(model.embed(corpus))          # embed every document ONCE

def vector_search(query):
    q = list(model.embed([query]))[0]         # embed the query
    scored = [(cosine(q, e), corpus[i]) for i, e in enumerate(doc_embs)]
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored

for q in ["puppy playing outside", "growing vegetables at home", "waterbirds"]:
    print(f"\nQuery: {q!r}")
    for score, text in vector_search(q):
        print(f"  {score:.3f}  {text}")

The corpus organizes itself by meaning — cat/dog docs cluster, garden sits across town:

In [ ]:
for i in range(len(corpus)):
    for j in range(i + 1, len(corpus)):
        print(f"doc{i} vs doc{j}: {cosine(doc_embs[i], doc_embs[j]):.3f}")

## Where vector search fails — exact identifiers

Two near-twin docs differing in one character. The model thinks they're nearly the same
sentence — and ranks the WRONG one first for the identifier query.

In [ ]:
idcorpus = [
    "Error E-4042 refund transaction declined by the payment gateway",
    "Error E-4043 refund transaction succeeded but receipt email failed",
    "How refunds work a general overview of the refund process",
]
idembs = list(model.embed(idcorpus))

print("doc A (E-4042) vs doc B (E-4043):", round(cosine(idembs[0], idembs[1]), 3))

q = list(model.embed(["error E-4042"]))[0]
print("\nQuery: 'error E-4042'")
for s, t in sorted(((cosine(q, e), t) for e, t in zip(idembs, idcorpus)), reverse=True):
    print(f"  {s:.3f}  {t}")

## PRODUCTION — a vector database does storage + ANN

At scale you don't loop over every document — Qdrant stores the embeddings and finds
nearest neighbors with HNSW. Same model, so the scores match our from-scratch loop exactly.

In [ ]:
from qdrant_client import QdrantClient, models

DENSE = "sentence-transformers/all-MiniLM-L6-v2"

client = QdrantClient(":memory:")                   # real server: QdrantClient(url=...)
client.create_collection(
    collection_name="vector_demo",
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)
client.upsert(
    collection_name="vector_demo",
    points=[
        models.PointStruct(id=i, vector=models.Document(text=t, model=DENSE), payload={"text": t})
        for i, t in enumerate(corpus)
    ],
)

hits = client.query_points(
    collection_name="vector_demo",
    query=models.Document(text="puppy playing outside", model=DENSE),
    limit=3,
)
for h in hits.points:
    print(f"  {h.score:.3f}  {h.payload['text']}")

## Two searchers, opposite blind spots

| Query looks like | Winner |
|---|---|
| `puppy playing outside` (meaning) | Vector search |
| `error E-4042` (exact identifier) | BM25 |

**Next notebook: `2c_hybrid_search.ipynb`** — run both and fuse the rankings.